## Parameter Selection

In [1]:
import math

In [2]:
def safe_log2(x):
    if x <= 0:
        return -float('inf')  # log(0) → effectively removes the term
    return math.log2(x)

In [3]:
def get_error(del_r, k,d,l=10, s_i=1, w=2, security_level=128, scheme_id=0, modulo_diff_power=4,t=3):
    
    """
    Input:  - del_r: the minimum degree of the polynomial.
            - k: the number of the parties.
            - d: the circuit depth to be evaluated.
            - s_i: the norm of the secret for one party.
            - l: the log2 of the moduli to start from.
            - w : the decomposition base.
            - modulo_diff_power: the difference between powers (in our case 4) 
            - scheme_id: the id of the scheme for which the estimation is happening.
             (scheme_id=0 means the one based on LWR, scheme_id=1 means the scheme based on LWR(introduced by us),
             scheme_id=2 means the one based on LWE.)
    Output: the parameter set that matches the required level of security.
    
    The error is given as a lower bound for p2 in the case of LWR and a lower bound of q for LWE
    """
    s_norm = k*s_i
    p2byp1 = 2**(-1*modulo_diff_power)
    if(scheme_id==0):
        l = l-(modulo_diff_power*2)
        #(base decomposition of p_2 with base 2)
        y_in = (p2byp1)*(del_r*s_norm) + 1/2
        c_1 = (del_r*s_norm + 4)*(del_r*t) + del_r/2
        c_2= (t**2)*del_r*(del_r*s_norm/2 + 2.5) + ((p2byp1**2)*(del_r**2 )*(s_norm**2)+ (p2byp1*del_r*s_norm) + 1)/2 + (l*w*k*(del_r**2)*s_norm*p2byp1)/2 + k*l*w*del_r/2
        d_e1 = (c_1**(d-1))*(c_1*y_in + c_2)
#         print(math.log2(d_e1))
        return (math.log2(d_e1)+security_level) ### for LWR (their construction) we need (2^{lambda} *(circuit_error))/(2**modulo_diff)<= (p2/t) i.e (2^{lambda} *(circuit_error))<= (p1/t)    
    if(scheme_id==1):
        l = l-(modulo_diff_power*2)
        y_in1 = (p2byp1 + 1)*(del_r*s_norm)/2 + 1/2
        c_11 = (del_r*s_norm + 4)*(del_r*t) + del_r/2
        c_21= (t**2)*del_r*(del_r*s_norm/2 + 2.5) + ((del_r**2 )*(s_norm**2)+ (del_r*s_norm) + 1)/2 + (3*l*w*k*(del_r**2)*s_norm*p2byp1)/4 + k*l*w*del_r/2 + del_r*s_norm/2
        d_e2 = (c_11**(d-1))*(c_11*y_in1 + c_21)
#         print(math.log2(d_e2))
        return (math.log2(d_e2)+security_level)  ### for LWR we need (2^{lambda} *(circuit_error))<= (p2/t)
    
    
    if(scheme_id==2):
        e_1 = 2**(modulo_diff_power-1) ### The corresponding error for LWE is 2**power/2
        y_in2 = 2*del_r*e_1*s_norm + e_1
        c_12 = (del_r*s_norm + 4)*(del_r*t) + del_r/2
        c_22= (t**2)*del_r*(del_r*s_norm/2 + 2.5) + ((del_r**2 )*(s_norm**2) + (del_r*s_norm) + 1)/2 + (3*l*w*k*(del_r**2)*s_norm*e_1) + 2*l*w*k*del_r*e_1*s_norm
        d_e3 = (c_12**(d-1))*(c_12*y_in2 + c_22)
#         print(math.log2(d_e3))
        return math.log2(d_e3)+security_level ### For LWE, we need 2^{\lambda}*(circuit_error)<=q/(2t)
    
    


    

In [4]:
import math

# -----------------------------
# Stable log2(a + b)
# -----------------------------
def log2_sum(a, b):
    if a > b:
        return a + safe_log2(1 + 2**(b - a))
    else:
        return b + safe_log2(1 + 2**(a - b))


# -----------------------------
# Log-domain error function
# -----------------------------
def get_error_log(del_r, k, d, l=10, s_i=1, w=2,
                  security_level=128, scheme_id=0,
                  modulo_diff_power=4, t=3):

    s_norm = k * s_i
    p2byp1 = 2**(-modulo_diff_power) 
    log_del_r = safe_log2(del_r)
    log_t = safe_log2(t)
    log_s_norm = safe_log2(s_norm) if s_norm > 0 else -float('inf')

    log_p2byp1 = -modulo_diff_power  # log2(2^{-modulo_diff_power})

    # -----------------------------
    # Common c1 (all schemes)
    # c1 = (del_r*s_norm + 4)*(del_r*t) + del_r/2
    # -----------------------------
    log_term1 = safe_log2(del_r * s_norm + 4)
    log_term2 = log_del_r + log_t
    log_part1 = log_term1 + log_term2
    log_part2 = log_del_r - 1
    log_c1 = log2_sum(log_part1, log_part2)

    # =============================
    # Scheme 0
    # =============================
    if scheme_id == 0:

        l = l - 2 * modulo_diff_power

        # y_in = (p2/p1)*(del_r*s_norm) + 1/2
        log_y1 = log_p2byp1 + log_del_r + log_s_norm
        log_y2 = -1
        log_y_in = log2_sum(log_y1, log_y2)

        # ---- c2 ----
        # term A: t^2 * del_r * (del_r*s_norm/2 + 2.5)
        log_A1 = 2*log_t + log_del_r
        log_A2 = safe_log2(del_r*s_norm/2 + 2.5)
        log_A = log_A1 + log_A2

        # term B: ((p2/p1)^2 * del_r^2 * s_norm^2)
        log_B = 2*log_p2byp1 + 2*log_del_r + 2*log_s_norm

        # term C: (p2/p1 * del_r*s_norm)
        log_C = log_p2byp1 + log_del_r + log_s_norm

        # term D: constant 1
        log_D = 0

        log_inner = log2_sum(log_B, log_C)
        log_inner = log2_sum(log_inner, log_D)
        log_inner -= 1  # division by 2

        # term E: l*w*k*(del_r^2)*s_norm*p2byp1 / 2
        log_E = safe_log2(l*w*k) + 2*log_del_r + log_s_norm + log_p2byp1 - 1

        # term F: k*l*w*del_r/2
        log_F = safe_log2(k*l*w) + log_del_r - 1

        log_c2 = log2_sum(log_A, log_inner)
        log_c2 = log2_sum(log_c2, log_E)
        log_c2 = log2_sum(log_c2, log_F)

        # ---- final ----
        log_c1_y = log_c1 + log_y_in
        log_sum = log2_sum(log_c1_y, log_c2)

        log_d_e = (d-1)*log_c1 + log_sum

#         return log_d_e + security_level - modulo_diff_power
        return log_d_e + security_level 


    # =============================
    # Scheme 1
    # =============================
    if scheme_id == 1:

        l = l - 2 * modulo_diff_power

        # y_in1
        log_y_main = safe_log2(p2byp1 + 1) + log_del_r + log_s_norm - 1
        log_y_in = log2_sum(log_y_main, -1)

        # c2 similar structure
        log_A = 2*log_t + log_del_r + safe_log2(del_r*s_norm/2 + 2.5)

        log_B = 2*log_del_r + 2*log_s_norm
        log_C = log_del_r + log_s_norm
        log_D = 0

        log_inner = log2_sum(log_B, log_C)
        log_inner = log2_sum(log_inner, log_D)
        log_inner -= 1

        log_E = safe_log2(3*l*w*k/4) + 2*log_del_r + log_s_norm + log_p2byp1
        log_F = safe_log2(k*l*w) + log_del_r - 1
        log_G = log_del_r + log_s_norm - 1

        log_c2 = log2_sum(log_A, log_inner)
        log_c2 = log2_sum(log_c2, log_E)
        log_c2 = log2_sum(log_c2, log_F)
        log_c2 = log2_sum(log_c2, log_G)

        log_c1_y = log_c1 + log_y_in
        log_sum = log2_sum(log_c1_y, log_c2)

        log_d_e = (d-1)*log_c1 + log_sum

        return log_d_e + security_level


    # =============================
    # Scheme 2 (LWE)
    # =============================
    if scheme_id == 2:

        log_e1 = modulo_diff_power - 1

        # y_in2
        log_y1 = safe_log2(2) + log_del_r + log_e1 + log_s_norm
        log_y_in = log2_sum(log_y1, log_e1)

        # c2
        log_A = 2*log_t + log_del_r + safe_log2(del_r*s_norm/2 + 2.5)

        log_B = 2*log_del_r + 2*log_s_norm
        log_C = log_del_r + log_s_norm
        log_D = 0

        log_inner = log2_sum(log_B, log_C)
        log_inner = log2_sum(log_inner, log_D)
        log_inner -= 1

        log_E = safe_log2(3*l*w*k) + 2*log_del_r + log_s_norm + log_e1
        log_F = safe_log2(2*l*w*k) + log_del_r + log_e1 + log_s_norm

        log_c2 = log2_sum(log_A, log_inner)
        log_c2 = log2_sum(log_c2, log_E)
        log_c2 = log2_sum(log_c2, log_F)

        log_c1_y = log_c1 + log_y_in
        log_sum = log2_sum(log_c1_y, log_c2)

        log_d_e = (d-1)*log_c1 + log_sum

        return log_d_e + security_level

In [5]:
"""get d in the log domain"""
def get_d(log2_modulo, del_r, k, l, s_i=1, w=2,
          security_level=128, scheme_id=0,
          modulo_diff_power=4, t=3):

    s_norm = k * s_i

   
    if scheme_id == 2 or scheme_id ==1:
        modulo_diff_power = 0

   
    U = log2_modulo - math.log2(2 * t) - security_level + modulo_diff_power

    p2byp1 = 2**(-modulo_diff_power)

    # -----------------------------
    # Scheme 0
    # -----------------------------
    if scheme_id == 0:
        U = U -2*modulo_diff_power

        l = l - (modulo_diff_power * 2)

        y_in = (p2byp1) * (del_r * s_norm) + 0.5

        c1 = (del_r * s_norm + 4) * (del_r * t) + del_r / 2

        c2 = (
            (t**2) * del_r * (del_r * s_norm / 2 + 2.5)
            + ((p2byp1**2) * (del_r**2) * (s_norm**2)
               + (p2byp1 * del_r * s_norm) + 1) / 2
            + (l * w * k * (del_r**2) * s_norm * p2byp1) / 2
            + k * l * w * del_r / 2
        )

        A = c1 * y_in + c2

    
        d = (U - math.log2(A)) / math.log2(c1) + 1

        return max(0, round(d))


    # -----------------------------
    # Scheme 1
    # -----------------------------
    if scheme_id == 1:
        U = U -2*modulo_diff_power

        l = l - (modulo_diff_power * 2)

        y_in = (p2byp1 + 1) * (del_r * s_norm) / 2 + 0.5

        c1 = (del_r * s_norm + 4) * (del_r * t) + del_r / 2

        c2 = (
            (t**2) * del_r * (del_r * s_norm / 2 + 2.5)
            + ((del_r**2) * (s_norm**2) + (del_r * s_norm) + 1) / 2
            + (3 * l * w * k * (del_r**2) * s_norm * p2byp1) / 4
            + k * l * w * del_r / 2
            + del_r * s_norm / 2
        )

        A = c1 * y_in + c2

        d = (U - math.log2(A)) / math.log2(c1) + 1

        return max(0, round(d))


    # -----------------------------
    # Scheme 2 (LWE)
    # -----------------------------
    if scheme_id == 2:

        e1 = 2**(modulo_diff_power - 1)

        y_in = 2 * del_r * e1 * s_norm + e1

        c1 = (del_r * s_norm + 4) * (del_r * t) + del_r / 2

        c2 = (
            (t**2) * del_r * (del_r * s_norm / 2 + 2.5)
            + ((del_r**2) * (s_norm**2) + (del_r * s_norm) + 1) / 2
            + (3 * l * w * k * (del_r**2) * s_norm * e1)
            + 2 * l * w * k * del_r * e1 * s_norm
        )

        A = c1 * y_in + c2

        d = (U - math.log2(A)) / math.log2(c1) + 1

        return max(0, round(d))

In [ ]:
import math
from LWE_estimator.estimator import *

# -----------------------------
# Global settings
# -----------------------------
modulo_diff_power = 4
parameters = {}

Xs = ND.UniformMod(3)
Xe = ND.DiscreteGaussian(4.61)

# -----------------------------
# Cache for security evaluations
# -----------------------------
security_cache = {}

def compute_security(log_n, log_modulo):
    key = (log_n, log_modulo)
    if key in security_cache:
        return security_cache[key]

    l = LWE.Parameters(
        n=2**log_n,
        q=2**log_modulo,
        Xs=Xs,
        Xe=Xe
    )

    sec_list = [
        LWE.primal_bdd(l),
        LWE.primal_usvp(l),
        LWE.dual_hybrid(l)
    ]

    sec_level = min(math.log2(alg['rop']) for alg in sec_list)
    security_cache[key] = sec_level

    return sec_level


# -----------------------------
# Compute minimal q from error
# -----------------------------
def compute_logq_from_error(log_n, k, d, scheme_id, t=3, w=2, target_level=128):

    log2_error = get_error_log(
        2**log_n,
        k,
        d,
        0,  # q not needed for error growth
        s_i=1,
        w=w,
        security_level=target_level,
        scheme_id=scheme_id
    )

    return math.ceil(log2_error + math.log2(2*t))


# -----------------------------
# Feasibility check
# -----------------------------
def is_feasible(log_n, k, d, scheme_id, target_level=128):

    log_q = compute_logq_from_error(log_n, k, d, scheme_id)

    sec = compute_security(log_n, log_q)

    return sec >= target_level, log_q, sec


# -----------------------------
# Binary search for max d
# -----------------------------
def find_max_d(log_n, k, scheme_id, target_level=128):

    low = 1
    high = 100  # you can increase if needed

    best_d = -1
    best_q = None
    best_sec = None

    while low <= high:

        mid = (low + high) // 2

        feasible, log_q, sec = is_feasible(
            log_n, k, mid, scheme_id, target_level
        )

        if feasible:
            best_d = mid
            best_q = log_q
            best_sec = sec
            low = mid + 1   # try larger d
        else:
            high = mid - 1  # reduce d

    return best_d, best_q, best_sec


# -----------------------------
# Main loop
# Sheme 0 indicates LWE, Scheme 1 indicates LWR(construction 1)
# Scheme 2 indicates LWR (construction 2)
# -----------------------------
for scheme_id in [0,1,2]:

    for log_n in range(13, 18):

        for k in [2, 4, 8, 128]:

            d, log_q, sec = find_max_d(log_n, k, scheme_id)

            if d == -1:
                print(f"No valid params for log_n={log_n}, k={k}, scheme={scheme_id}")
                continue

            if scheme_id in [0,1]:
                result = (
                    log_n,
                    log_q,
                    log_q + modulo_diff_power,
                    log_q + 2*modulo_diff_power,
                    d,
                    k,
                    sec
                )
            else:
                result = (
                    log_n,
                    log_q,
                    d,
                    k,
                    sec
                )

            parameters[(log_n, 128, scheme_id, k, d)] = result

            print("Selected:", result)


# -----------------------------
# Final output
# -----------------------------
print("\nFinal parameters:")
for key, val in parameters.items():
    print(f"{key} -> {val}")

Selected: (13, 198, 202, 206, 2, 2, 139.63978909660162)
Selected: (13, 201, 205, 209, 2, 4, 137.41040954106674)
Selected: (13, 204, 208, 212, 2, 8, 135.2141869023522)
Selected: (13, 182, 186, 190, 1, 128, 153.01173285169546)
Selected: (14, 417, 421, 425, 9, 2, 132.66574008342403)
Selected: (14, 427, 431, 435, 9, 4, 129.52134656368034)
Selected: (14, 405, 409, 413, 8, 8, 136.9686764952472)
Selected: (14, 404, 408, 412, 7, 128, 137.31675714341563)
Selected: (15, 860, 864, 868, 22, 2, 129.22126871464755)
Selected: (15, 849, 853, 857, 21, 4, 131.087189269316)
Selected: (15, 837, 841, 845, 20, 8, 133.04132173035126)
Selected: (15, 844, 848, 852, 18, 128, 131.91714299658202)
Selected: (16, 1735, 1739, 1743, 46, 2, 129.04420342375866)
Selected: (16, 1746, 1750, 1754, 45, 4, 128.21138554361093)
Selected: (16, 1719, 1723, 1727, 43, 8, 130.189649397052)
Selected: (16, 1733, 1737, 1741, 39, 128, 129.10484557274916)
Selected: (17, 3511, 3515, 3519, 92, 2, 128.3924374492024)
Selected: (17, 3491, 34